# footprint: vendor AI public-footprint demo

**Confidential case-study material.** Never share or publish this notebook's outputs, the bundle or the exported workbook. The tool reads public sources only and never contacts a vendor.

The notebook drives the same engine as the CLI and the Streamlit app, in eight steps:

1. **Setup**: load the code and the frozen evidence pack.
2. **Upload**: the vendor workbook.
3. **Criticality** (columns L and M), from the vendor profile only.
4. **Assessment depth** (column N).
5. **Run**: replay by default, which is offline and reproducible. A live run of one vendor is optional.
6. **Results**: verdicts, evidence, risk and coverage.
7. **Export and download**: the completed workbook.
8. **Verify**: workbook fidelity, cell equality and evidence re-verification.

**Local Jupyter.** Run `uv sync --all-extras`, then `uv run jupyter notebook notebooks/footprint_demo.ipynb`. The notebook uses this checkout and its `.env`.

**Google Colab.**
- On the development machine, build the bundle with `uv run python -c "from footprint.bundle import build_bundle; print(build_bundle())"`. Keep `footprint_bundle.zip` private, and never commit it.
- In Colab, open this notebook (File > Upload notebook) and run the setup cell. Choose the zip when asked. No GitHub token is needed.
- Gemini is optional, and only `live_ai` uses it. To enable it, add `GEMINI_API_KEY` under Colab **Secrets** (the key icon). Never paste a key into a cell.

In [ ]:
# Parameters: edit, then run all cells (the tests override this cell).
MODE = "replay"            # "replay" (default): offline, from the frozen evidence pack and caches
                           # "live_rules" or "live_ai": polite live collection of ONE vendor (set VENDORS)
VENDORS = None             # e.g. ["V-005"]; None = every vendor in the workbook
AS_OF = None               # ISO date; None = the date of the frozen collection runs (replay) or today (live)
WORKBOOK = "data/input/Meridian_Vendor_Input.xlsx"  # used when nothing is uploaded
ASK_UPLOAD = True          # Colab: file picker; local Jupyter: an upload button (else WORKBOOK is used)
TEAM = None                # column V; None = FOOTPRINT_TEAM_NAME from .env (local) or Colab secrets
LOAD_SECRETS = True        # local: read .env; Colab: read Colab secrets. Values are never shown
SHOW_VENDOR = "V-005"      # the vendor whose full L-V cells the results show
OUT_DIR = "scratch/notebook"  # the exported workbook goes here (git-ignored)
REFERENCE = None           # optional: an earlier export; verify then checks that replay reproduced its L-V cells

## 1. Setup

- **Colab, first run:** choose `footprint_bundle.zip` when asked. The cell installs the bundled wheel and checks every file against the bundle's SHA-256 manifest.
- **Local Jupyter:** the cell uses this repository.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.isdir("/content/footprint/src"):  # first run: upload, unpack and install the bundle
    import zipfile
    from google.colab import files
    print("Choose footprint_bundle.zip (private: never share or commit it).")
    zipfile.ZipFile(next(iter(files.upload()))).extractall("/content/footprint")
    %pip install -q /content/footprint/dist/footprint-*.whl
if IN_COLAB and "/content/footprint/src" not in sys.path:
    sys.path.insert(0, "/content/footprint/src")  # the bundled code finds config/, prompts/ and evidence/ beside it
    os.chdir("/content/footprint")

from footprint.bundle import notebook_setup

env = notebook_setup(mode=MODE, vendors=VENDORS, team=TEAM, load_secrets=LOAD_SECRETS)
env

## 2. Upload the vendor workbook

The reader tolerates header typos (fuzzy matching) and skips the V-000 worked example. Only the cream student cells L-V of the vendor rows are ever written, and always to a new file.

In [ ]:
from footprint.bundle import WorkbookInput

workbook = WorkbookInput(WORKBOOK, ask=ASK_UPLOAD)
workbook

## 3. Criticality (columns L and M)

A deterministic rubric scores five factors from the profile alone: O×7, D×6, P×5, R×4 and V×3, with floors F1-F6. Public evidence never changes the tier. Analyst overrides (HC1) come from the review store, and each one needs a reason.

In [ ]:
from footprint.bundle import criticality_frame, read_inventory, tier_overrides
from footprint.pipeline import assess_example, assess_profiles

data = read_inventory(workbook)
assessments = assess_profiles(data, tier_overrides())
criticality_frame(assessments, example=assess_example(data))

In [ ]:
from footprint.bundle import rationale_table

rationale_table(assessments)

## 4. Assessment depth (column N)

The tier sets the mandatory source families, the request caps and the budgets. Modifiers deepen specific families. Steps that need the vendor (questionnaire, contract review, attestations) are reserved for Meridian.

In [ ]:
from footprint.bundle import depth_frame

depth_frame(assessments)

## 5. Run the assessment

- **Replay** (the default) makes no network calls. It rebuilds every finding from the frozen evidence pack, the LLM cache and the review stores, so the cells are reproducible.
- **Live modes** collect one vendor politely: ToS register, then robots.txt, then a rate limit, with GET requests only. `live_ai` also sends public passages to Gemini through the payload guard, and never any Meridian data.

In [ ]:
from footprint.bundle import progress_printer
from footprint.pipeline import review_stores, run_assessment

overrides, reviews = review_stores()  # review/overrides.jsonl and review/reviews.jsonl, when they exist
result = run_assessment(workbook.source(), MODE, vendors=VENDORS, as_of=AS_OF, team=env.team,
                        overrides=overrides, reviews=reviews, progress=progress_printer())
result.run_id, result.mode, result.as_of

## 6. Results

The tables below are, in order:
- the answer for each vendor: tier, AI usage, risk and flip condition;
- the cited evidence, with its tags;
- the risk steps;
- the Coverage Log, which records negative evidence;
- the full L-V cells for one vendor.

In [ ]:
from footprint.bundle import summary_frame

summary_frame(result)

In [ ]:
from footprint.bundle import evidence_frame

evidence_frame(result)

In [ ]:
from footprint.bundle import risk_frame

risk_frame(result)

In [ ]:
from footprint.bundle import coverage_frame

coverage_frame(result)

In [ ]:
from footprint.bundle import cells_table

cells_table(result, SHOW_VENDOR)

## 7. Export and download

The export writes L-V for every vendor and appends these sheets:
- Evidence Log
- Coverage Log
- Criticality Workings
- Method & Legend
- Evidence Images
- Run Info

The input workbook is never modified.

In [ ]:
from footprint.bundle import offer_download, output_path
from footprint.pipeline import export_assessment

out_path = output_path(OUT_DIR, result)
report = export_assessment(result, workbook.source(), out_path, env.team)
offer_download(out_path)

## 8. Verify

The checks confirm three things:
- the export changed nothing outside L-V and the appended sheets, so V-000 and the provided cells are untouched;
- the exported cells equal the assessment;
- every cited excerpt still re-verifies against its captured source (SHA-256 and exact quote).

In [ ]:
from footprint.bundle import verify_export

checks = verify_export(result, workbook.source(), out_path, reference=REFERENCE)
checks